# AST AudioSet — DIMER audio event classification tutorial

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/ast-audio-classification-pipeline)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/ast-audio-classification-pipeline/blob/main/tutorials/ast_audio_classification_colab.ipynb)
[![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-MIT%2Fast--finetuned--audioset--10--10--0.4593-ffcc4d?style=flat)](https://huggingface.co/MIT/ast-finetuned-audioset-10-10-0.4593)
[![Upstream](https://img.shields.io/badge/Upstream-YuanGongND%2Fast-181717?style=flat&logo=github&logoColor=white)](https://github.com/YuanGongND/ast)
[![arXiv](https://img.shields.io/badge/arXiv-2104.01778-b31b1b.svg)](https://arxiv.org/abs/2104.01778)

**Profile:** `TASK-INFERENCE`
**Notebook specification:** DIMER Notebook Specification 1.0
**Capability:** multi-label audio event classification over the 527 AudioSet labels using the pinned `MIT/ast-finetuned-audioset-10-10-0.4593` weights

This notebook is the executable reference path for the repository capability. It exercises the repository's public pipeline API rather than reimplementing model inference. At inference the feature extractor turns a 16 kHz mono waveform into a 128-bin Kaldi filterbank padded or cropped to 1,024 frames (10.24 s), the Audio Spectrogram Transformer encoder emits one logit per AudioSet label, and the pipeline applies an independent **sigmoid** to each logit and ranks the labels by score. **No adaptation occurs:** no training, fine-tuning, in-context conditioning, or preprocessing fitting happens in this notebook — the upstream checkpoint supplies the weights and the feature-extractor configuration, and this repository adds packaging, snapshot verification, input validation with named ceilings, the explicit sigmoid with top-k selection, and provenance fields. The default sample is a synthetic tone generated in code; its ranking is demonstration (plumbing) evidence, not a production-quality or benchmark claim.

**Learning objectives:** bootstrap the repository in a fresh runtime, resolve the immutable upstream model revision, generate and validate a synthetic default input against the pipeline's ceilings (sample rate, model window, input ceiling), run the supported task, read multi-label sigmoid scores correctly (independent per label, uncalibrated, no shipped threshold), exercise an optional BYOD path with a PCM WAV file, and export machine-readable outputs plus provenance.

**This notebook does not demonstrate:** speech transcription, speaker identification, temporal localisation of events inside the window, source separation, detection of sounds outside the 527 AudioSet labels, or any training. It also reports no accuracy metric: the repository ships no metric helper and no labelled audio.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). The default path runs on CPU and uses CUDA automatically when available; inference is float32 on both. The pinned `torch==2.14.0` install is the largest download of the run, followed by the ~346 MB checkpoint.
- **Knowledge:** basic Python and NumPy; what a sigmoid score over independent labels means (multi-label, not a distribution over classes).
- **Data:** the default sample is a deterministic 3 s, 440 Hz sine tone generated in code at 16 kHz, so nothing is downloaded and no private data is needed. Optional BYOD upload is gated off by default so the sample path can run top-to-bottom without interaction. Expected BYOD input: one PCM WAV file (8-, 16- or 32-bit integer samples; mono or stereo — stereo is averaged to mono), any sample rate (the pipeline resamples to 16 kHz), between 0.025 s and 120 s long; only the first 10.24 s reach the model. Do not upload confidential or restricted data to a hosted notebook environment unless you are authorized to do so. Uploaded inputs remain in the notebook runtime; this pipeline does not send them to a third-party inference API.
- **External access:** GitHub (repository clone) and the Hugging Face Hub (the package's `stage_missing_files` fetches the pinned checkpoint, ~346 MB, once, because the Git repository does not vendor the weights). No credentials are required.

## 1. Bootstrap the repository and pinned runtime

When the notebook is opened without a repository checkout, this cell clones the repository. Released notebooks default to `main`; automated candidate validation can set `DIMER_TUTORIAL_REF` to an immutable commit or review branch. The repository is installed as a regular (non-editable) package so it is importable in this same runtime; an editable install would only become importable after a restart. Model-facing dependencies (`torch`, `torchaudio`, `transformers`, `safetensors`, `numpy`) are pinned exactly in `pyproject.toml`. If installation replaces any package that this runtime has already imported, the cell fails with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the repository revision, Python, `torch`, `torchaudio` and `transformers` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/kurtvalcorza/ast-audio-classification-pipeline.git'
REPO_NAME = 'ast-audio-classification-pipeline'
REPO_REF = os.environ.get('DIMER_TUTORIAL_REF', 'main')
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'
ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    checkout = ROOT / REPO_NAME
    if not checkout.exists():
        subprocess.run(['git', 'clone', '--filter=blob:none', '-q', REPO_URL, str(checkout)], check=True)
    if REPO_REF != 'main':
        subprocess.run(['git', '-C', str(checkout), 'fetch', '--depth', '1', 'origin', REPO_REF], check=True)
        subprocess.run(['git', '-C', str(checkout), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
    else:
        subprocess.run(['git', '-C', str(checkout), 'checkout', '-q', 'main'], check=True)
        subprocess.run(['git', '-C', str(checkout), 'pull', '--ff-only', '-q', 'origin', 'main'], check=True)
    os.chdir(checkout)
    ROOT = Path.cwd()

if not SKIP_INSTALL:
    # Every distribution that is already imported in this runtime is captured before installation,
    # whatever its name (PIL -> pillow), so a pinned install that replaces any loaded package is
    # detected. Distribution metadata is compared with metadata afterwards: torch.__version__ carries
    # a local build label (for example 2.14.0+cu130) that the distribution version omits.
    def _installed_version(distribution):
        try:
            return importlib.metadata.version(distribution)
        except importlib.metadata.PackageNotFoundError:
            return None
    _module_dists = importlib.metadata.packages_distributions()
    _loaded_dists = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded_dists}
    # Non-editable install: an editable (.pth) install is not importable until the
    # interpreter restarts, which a fresh hosted runtime cannot do mid-notebook.
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', str(ROOT)], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

REPO_SHA = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
import platform, torch, torchaudio, transformers
print({'repository': str(ROOT), 'repository_revision': REPO_SHA, 'requested_ref': REPO_REF, 'python': platform.python_version(), 'torch': torch.__version__, 'torchaudio': torchaudio.__version__, 'transformers': transformers.__version__, 'cuda': torch.cuda.is_available()})

## 2. Generate the synthetic sample or optional BYOD

The default sample is **synthetic**: a deterministic 3 s, 440 Hz sine tone at amplitude 0.5, generated in code at the model's 16 kHz rate as a float32 mono array (the same kind of input the repository's smoke run used), so it needs no download and its SHA-256 is printed for the record. A pure tone is not a recording of any real acoustic event, so it has **no ground truth** and whatever ranking the model returns is a sanity check that the input contract, feature extraction and forward pass work — not a correctness measurement. BYOD is optional and disabled by default; when enabled, upload one PCM WAV file. It is decoded with the standard-library `wave` module (no extra decoder is pinned), integer samples are scaled to `[-1, 1]`, stereo is averaged to mono, and the original sample rate is passed to the pipeline, which resamples to 16 kHz with `torchaudio.functional.resample` — resampling cannot restore content above the original Nyquist frequency.

Before anything expensive runs, this cell surfaces the pipeline's operational ceilings — `SAMPLE_RATE` (16,000 Hz), `MAX_AUDIO_SECONDS` (the 10.24 s model window), `MAX_INPUT_SECONDS` (120 s hard ceiling), `NUM_LABELS` (527) — and checks the clip against them with clear messages: a clip above 120 s is rejected here (chunk it first), and a clip longer than 10.24 s is accepted but **only its first 10.24 s reach the model**; the notebook says so before inference and the result carries `truncated: True`. Clips shorter than one 25 ms filterbank frame are rejected by the pipeline. Nothing else is dropped or altered. Look for a dictionary naming the sample kind, its duration, sample rate, digest and whether it will be truncated.

In [ ]:
import hashlib
import io
import wave

import numpy as np

from ast_audio_classification_pipeline import MAX_AUDIO_SECONDS, MAX_INPUT_SECONDS, NUM_LABELS, SAMPLE_RATE

USE_BYOD = False  # @param {type:"boolean"}
TONE_SECONDS = 3.0
TONE_HZ = 440.0
TONE_AMPLITUDE = 0.5

print({'ceilings': {'SAMPLE_RATE': SAMPLE_RATE, 'MAX_AUDIO_SECONDS': MAX_AUDIO_SECONDS, 'MAX_INPUT_SECONDS': MAX_INPUT_SECONDS, 'NUM_LABELS': NUM_LABELS}})
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    clip_name = next(iter(uploaded))
    with wave.open(io.BytesIO(uploaded[clip_name]), 'rb') as handle:
        channels, sample_width, sample_rate, frames = handle.getnchannels(), handle.getsampwidth(), handle.getframerate(), handle.getnframes()
        raw = handle.readframes(frames)
    if sample_width == 1:
        samples = (np.frombuffer(raw, dtype=np.uint8).astype(np.float32) - 128.0) / 128.0
    elif sample_width == 2:
        samples = np.frombuffer(raw, dtype='<i2').astype(np.float32) / 32768.0
    elif sample_width == 4:
        samples = np.frombuffer(raw, dtype='<i4').astype(np.float32) / 2147483648.0
    else:
        raise ValueError(f'{clip_name}: {8 * sample_width}-bit PCM is not supported here; convert the file to 16-bit PCM WAV and rerun this cell.')
    audio = samples.reshape(-1, channels).mean(axis=1).astype(np.float32) if channels > 1 else samples
    sample_kind = 'BYOD'
else:
    # Deterministic synthetic tone: no randomness, so no seed is needed and the digest is stable.
    sample_rate = SAMPLE_RATE
    t = np.arange(int(TONE_SECONDS * sample_rate)) / sample_rate
    audio = (TONE_AMPLITUDE * np.sin(2 * np.pi * TONE_HZ * t)).astype(np.float32)
    clip_name = f'synthetic_sine_{int(TONE_HZ)}hz_{int(TONE_SECONDS)}s.wav'
    sample_kind = 'synthetic'

duration_seconds = audio.shape[0] / sample_rate
if duration_seconds > MAX_INPUT_SECONDS:
    raise ValueError(f'{clip_name}: {duration_seconds:.2f} s exceeds MAX_INPUT_SECONDS={MAX_INPUT_SECONDS} s; split the clip into shorter chunks and rerun this cell.')
will_truncate = duration_seconds > MAX_AUDIO_SECONDS
if will_truncate:
    print(f'NOTE: {clip_name} is {duration_seconds:.2f} s; only the first {MAX_AUDIO_SECONDS} s reach the model (the rest is cropped and the result is flagged truncated).')
audio_sha256 = hashlib.sha256(audio.tobytes()).hexdigest()
print({'sample_kind': sample_kind, 'name': clip_name, 'duration_seconds': round(duration_seconds, 3), 'sample_rate': sample_rate, 'dtype': str(audio.dtype), 'float32_sha256': audio_sha256, 'will_truncate': will_truncate, 'will_resample': sample_rate != SAMPLE_RATE})

## 3. Stage, verify and resolve the pinned model

Model acquisition goes through the package, not the notebook. The public API pins the exact upstream revision (`MODEL_ID`/`MODEL_REVISION` are imported from the package, never typed here). The Git repository carries `weights/ast-audioset/dimer-base-manifest.json`, `config.json` and `preprocessor_config.json` but git-ignores the 346 MB `model.safetensors`, so in a fresh clone `stage_missing_files(WEIGHTS_DIR, allow_download=True)` fetches exactly the manifest entries that are absent, at the pinned revision, into the snapshot directory — it prints the list it fetched (`[]` on a warm runtime) and refuses a manifest whose identity differs from the package pins. `verify_snapshot(WEIGHTS_DIR)` then re-hashes every manifest entry (size and SHA-256) and raises on the first mismatch; its returned manifest dict is printed. Only then does `from_pretrained(weights_dir=WEIGHTS_DIR)` load the verified files with `local_files_only=True` and `trust_remote_code=False` — there is no fallback to a different download. The effective model identity, the device chosen (`cuda:0` when available, else `cpu`) and the label count are printed before inference.

In [ ]:
from ast_audio_classification_pipeline import MODEL_ID, MODEL_KEY, MODEL_REVISION, ASTAudioClassificationPipeline, stage_missing_files, verify_snapshot
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'num_labels': NUM_LABELS})
WEIGHTS_DIR = ROOT / 'weights' / MODEL_KEY
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
print(snapshot)
pipe = ASTAudioClassificationPipeline.from_pretrained(weights_dir=WEIGHTS_DIR)
print({'device': pipe.device, 'source': 'local-snapshot', 'labels': len(pipe.labels)})

## 4. Classify and read the scores correctly

`predict` returns a `predictions` list of `{label, index, score}` entries **ordered by descending score** — rank position is the label ordering, and the exported files preserve it — plus `activation` (`sigmoid`), `truncated`, `duration_seconds`, `window_seconds`, `resampled`, `input_sample_rate`, and the model identity. Each `score` is an **independent sigmoid** of that label's logit: this is multi-label classification, so the scores do not sum to one, several labels can be high at once, and a score is **not a calibrated probability** (the head was trained with binary cross-entropy on weak, incomplete AudioSet labels). **No decision threshold is shipped**: `top_k` (default 5, ceiling 527) is a presentation choice, not an acceptance rule, and no label is asserted present or absent. The operator owns the threshold and should set it per label from precision-recall curves on their own labelled clips; downstream calibration is the caller's responsibility.

No metric is reported in this notebook: the repository ships no metric helper and no labelled audio, and a synthetic tone has no ground truth. Measuring quality needs clips labelled against the same 527-label ontology, `predict(..., top_k=527)` for the full score vector, and mean average precision plus per-label precision at the chosen threshold; the "0.4593" in the checkpoint name is the upstream-reported AudioSet mAP for this configuration and is not measured here. Inference is deterministic on a fixed device and dtype (no sampling, `model.eval()`, `torch.inference_mode`); CUDA kernel selection and the resampler can shift scores in the third or fourth decimal place across hardware. Look for the ranked top-5 list. As recorded in the model card, the repository's smoke run on this same tone (CUDA, float32, verified snapshot) ranked `Sine wave` first at score 0.84; that is one observation for a synthetic tone and sanity evidence only — a materially different top label on your runtime is a signal to check the install, not a measurement of anything.

In [ ]:
result = pipe.predict(audio, sample_rate=sample_rate, top_k=5)
print({'activation': result['activation'], 'truncated': result['truncated'], 'resampled': result['resampled'], 'duration_seconds': round(result['duration_seconds'], 3), 'window_seconds': result['window_seconds'], 'device': pipe.device})
for rank, item in enumerate(result['predictions'], start=1):
    print(f"{rank:>2}. index {item['index']:>3}  score {item['score']:.4f}  {item['label']}")
metrics = {}
print('No metric is computed: the pipeline ships no metric helper and the sample has no ground truth; the ranking above is sanity evidence only.')

## 5. Export outputs and provenance

Machine-readable JSON preserves the full result (rank-ordered sigmoid scores, activation, truncation and resampling flags), the empty metrics block, the clip identity and digest, the repository revision, the model identifier, the immutable model revision, and the runtime identity (Python, `torch`, `torchaudio`, `transformers`, device). The rank-ordered top-k table is also written as CSV with explicit `clip`, `rank`, `index`, `label` and `score` columns so label ordering survives downstream use. No credentials are recorded.

In [ ]:
import csv
import json
os.makedirs('outputs', exist_ok=True)
payload = {
    'prediction': result,
    'metrics': metrics,
    'sample': {'kind': sample_kind, 'name': clip_name, 'duration_seconds': duration_seconds, 'sample_rate': sample_rate, 'float32_sha256': audio_sha256},
    'repository_revision': REPO_SHA,
    'model_id': MODEL_ID,
    'model_revision': MODEL_REVISION,
    'runtime': {
        'python': platform.python_version(),
        'torch': torch.__version__,
        'torchaudio': torchaudio.__version__,
        'transformers': transformers.__version__,
        'device': pipe.device,
    },
}
with open('outputs/ast_audio_classification_result.json', 'w', encoding='utf-8') as handle:
    json.dump(payload, handle, indent=2, ensure_ascii=False)
with open('outputs/ast_audio_classification_top_k.csv', 'w', encoding='utf-8', newline='') as handle:
    writer = csv.writer(handle)
    writer.writerow(['clip', 'rank', 'index', 'label', 'score'])
    for rank, item in enumerate(result['predictions'], start=1):
        writer.writerow([clip_name, rank, item['index'], item['label'], f"{item['score']:.6f}"])
print(sorted(os.listdir('outputs')))

## Interpretation and limits

The ranked labels are independent sigmoid scores over the fixed 527-label AudioSet ontology; they are not probabilities of presence, they do not sum to one, and the pipeline ships no threshold, so nothing in this notebook asserts that an event is present or absent. On the synthetic tone the ranking is sanity evidence by construction and no metric is measured; a ranking shown for a BYOD clip is a single-clip observation for that recording and must not be generalized to a domain, microphone, or acoustic environment. Only the first 10.24 s of a clip reach the model, resampling from other rates loses content above the original Nyquist frequency, and sounds outside the ontology still receive some ranked label. The pipeline provides no transcription, speaker identity, temporal localisation, source separation, or training capability.

Successful execution proves that the recorded repository revision can acquire the pinned model, validate the demonstrated input, execute the public pipeline path, and emit the shown machine-readable outputs in the tested runtime. It does **not** establish benchmark superiority, deployment calibration, safety for high-consequence decisions, or production fitness on an unseen domain.

**Next experiments:** enable `USE_BYOD` with a short real recording (a door closing, a dog barking) and compare how the sigmoid scores spread across related ontology labels; change `TONE_HZ` and watch which tone-like labels (`Sine wave`, `Dial tone`, `Beep, bleep`) move; upload a clip longer than 10.24 s and confirm `truncated: True`, then chunk it yourself and score each window separately.

## References

- Repository README: `../README.md`
- Repository model card: `../MODEL_CARD.md`
- Weight provenance: `../docs/WEIGHTS.md`
- Upstream model: https://huggingface.co/MIT/ast-finetuned-audioset-10-10-0.4593
- Upstream code: https://github.com/YuanGongND/ast
- AST: Audio Spectrogram Transformer (Gong, Chung, Glass, 2021): https://arxiv.org/abs/2104.01778
- AudioSet ontology: https://research.google.com/audioset/